# 제품 이상여부 판별 프로젝트


## 1. 데이터 불러오기


### 필수 라이브러리


In [1]:
# !pip install xgboost
# !pip install catboost
import os
from pprint import pprint

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.svm import SVC
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.calibration import CalibratedClassifierCV

from tqdm import tqdm

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', None)

### 데이터 읽어오기


In [3]:
ROOT_DIR = "data"
RANDOM_STATE = 110

# Load data
train_data = pd.read_csv(os.path.join(ROOT_DIR, "train.csv"))
list(train_data)
print("The total number of colums: " + str(len(train_data.columns)))

The total number of colums: 464


## 1st Data Preprocessing

#### Identify those columns with NA included

In [4]:
na_cal = []

for i in train_data.columns:
    if (train_data[i].isna().sum()) > 0:
        na_cal.append(i)

print("Total number of columns with NA: " + str(len(na_cal)))

train_data[na_cal] = train_data[na_cal].fillna(0)

Total number of columns with NA: 286


#### Identify those columns with one value duplicated to every row

In [5]:
one_val_duplicated = []
for i in train_data.columns:
    if (train_data[i].nunique()) == 1:
        one_val_duplicated.append(i)
        
print("Total number of columns with only one val: " + str(len(one_val_duplicated)))

Total number of columns with only one val: 313


#### Identify those columns with unique values for every row

In [6]:
# "Number of unique entries = Num rows" ==> "Unique value for every row"
num_rows = len(train_data)
unique_every_row = []
for i in train_data.columns:
    if (train_data[i].value_counts().size == num_rows):
        unique_every_row.append(i)
        
print("Total number of columns with unique values for every row: " + str(len(unique_every_row)))

Total number of columns with unique values for every row: 0


In [7]:
multiple_types = []
for i in train_data.columns:
    if (len(set(train_data[i].apply(type))) > 1):
        multiple_types.append(i)

print(multiple_types)
print("Total number of columns with multiple datatypes: " + str(len(multiple_types)))

['HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam', 'GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave', 'GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2', 'HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2']
Total number of columns with multiple datatypes: 8


In [8]:
# Columns with "OK" and numbers are mixed --> to be processed ("OK" to NaN):
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2

# Columns with "OK" and NaN are mixed:
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Dam
#     GMES_ORIGIN_INSP_JUDGE_CODE Collect Result_AutoClave
#     GMES_ORIGIN_INSP_JUDGE_CODE Judge Value_AutoClave
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill1
#     HEAD NORMAL COORDINATE X AXIS(Stage1) Judge Value_Fill2

#### Automating to replace of "OK"s with null/NaN as instructed

In [9]:
ok_2_nan = ["HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Dam", 
            "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill1", 
            "HEAD NORMAL COORDINATE X AXIS(Stage1) Collect Result_Fill2"]

for i in ok_2_nan:
    train_data[i] = train_data[i].replace('OK', np.nan)
    train_data[i] = train_data[i].astype(float)

#### ACTION!!!!

In [10]:
train_data.drop(columns = one_val_duplicated, inplace = True)
train_data.drop(columns = unique_every_row, inplace = True)

print("Number of columns: " + str(len(train_data.columns)))

Number of columns: 151


### 언더 샘플링


데이타 불균형을 해결하기 위해 언더 샘플링을 진행합니다.


In [11]:
normal_ratio = 1.0  # 1.0 means 1:1 ratio

df_normal = train_data[train_data["target"] == "Normal"]
df_abnormal = train_data[train_data["target"] == "AbNormal"]

num_normal = len(df_normal)
num_abnormal = len(df_abnormal)
print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}")

df_normal = df_normal.sample(n=int(num_abnormal * normal_ratio), replace=False, random_state=RANDOM_STATE)
df_concat = pd.concat([df_normal, df_abnormal], axis=0).reset_index(drop=True)
df_concat.value_counts("target")

  Total: Normal: 38156, AbNormal: 2350


target
AbNormal    2350
Normal      2350
Name: count, dtype: int64

### 데이터 분할


In [12]:
df_train, df_val = train_test_split(
    df_concat,
    test_size=0.3,
    stratify=df_concat["target"],
    random_state=RANDOM_STATE,
)


def print_stats(df: pd.DataFrame):
    num_normal = len(df[df["target"] == "Normal"])
    num_abnormal = len(df[df["target"] == "AbNormal"])

    print(f"  Total: Normal: {num_normal}, AbNormal: {num_abnormal}" + f" ratio: {num_abnormal/num_normal}")


# Print statistics
print(f"  \tAbnormal\tNormal")
print_stats(df_train)
print_stats(df_val)

  	Abnormal	Normal
  Total: Normal: 1645, AbNormal: 1645 ratio: 1.0
  Total: Normal: 705, AbNormal: 705 ratio: 1.0


## 3. 모델 학습


In [13]:
features = []

for col in df_train.columns:
    try:
        df_train[col] = df_train[col].astype(int)
        features.append(col)
    except:
        continue

train_x = df_train[features]
train_y = df_train["target"]


In [14]:
features = []

for col in df_val.columns:
    try:
        df_val[col] = df_val[col].astype(int)
        features.append(col)
    except:
        continue

test_x = df_val[features]
test_y = df_val["target"]

### 모델 학습 (XGB)


In [15]:
le = LabelEncoder()
encoded_y = le.fit_transform(train_y)
encoded_y_test = le.fit_transform(test_y)
label_mapping = dict(zip(le.classes_, range(len(le.classes_))))

# Print the mapping
print("Label to Integer Mapping:", label_mapping)

Label to Integer Mapping: {'AbNormal': 0, 'Normal': 1}


In [16]:
# xgb_model = XGBClassifier(random_state=RANDOM_STATE)

# params = {'max_depth' : [5, 7, 10, 15, 20], 
#           'min_child_weight' : [1, 3, 5, 8, 10], 
#           'colsample_bytree' : [0.3, 0.5, 0.75],
#           'learning_rate' : [0.1, 0.15, 0.2],
#           'subsample' : [0.5, 0.75, 1],
#           'colsample_bytree' : [0.5, 0.75, 1]}

# gridcv = GridSearchCV(xgb_model, param_grid=params, cv=5)

# gridcv.fit(train_x, encoded_y)
# print(gridcv.best_params_)

In [17]:
pre_cal_xgbc = XGBClassifier(n_estimators=300, 
                     learning_rate=0.01, 
                     max_depth=3, 
                     min_child_weight=4,
                     subsample=1,
                     colsample_bytree=0.8, 
                     random_state=RANDOM_STATE,
                     n_jobs=-1,
                     gamma=0.3,
                     eval_metric='logloss')


In [18]:
xgbc = CalibratedClassifierCV(pre_cal_xgbc, method='isotonic', cv=5)
xgbc.fit(train_x, encoded_y)

CalibratedClassifierCV(cv=5,
                       estimator=XGBClassifier(base_score=None, booster=None,
                                               callbacks=None,
                                               colsample_bylevel=None,
                                               colsample_bynode=None,
                                               colsample_bytree=0.8,
                                               device=None,
                                               early_stopping_rounds=None,
                                               enable_categorical=False,
                                               eval_metric='logloss',
                                               feature_types=None, gamma=0.3,
                                               grow_policy=None,
                                               importance_type=None,
                                               interaction_constraints=None,
                                               learning_rate=0.01, max_bin=None,
                                               max_cat_threshold=None,
                                               max_cat_to_onehot=None,
                                               max_delta_step=None, max_depth=3,
                                               max_leaves=None,
                                               min_child_weight=4, missing=nan,
                                               monotone_constraints=None,
                                               multi_strategy=None,
                                               n_estimators=300, n_jobs=-1,
                                               num_parallel_tree=None,
                                               random_state=110, ...),
                       method='isotonic')

### 모델 학습 (rf)


#### Hyperparameter Tuning

In [19]:
# params = { 'n_estimators' : [10, 30, 50, 80, 100, 150],
#            'max_depth' : [6, 8, 10, 12, 15, 20],
#            'min_samples_leaf' : [8, 12, 18, 20],
#            'min_samples_split' : [8, 16, 20, 25],
#            'n_jobs' : [-1],
#            'random_state' : [RANDOM_STATE],
#             }

# # RandomForestClassifier 객체 생성 후 GridSearchCV 수행
# rf_clf = RandomForestClassifier(random_state = 0, n_jobs = -1)
# grid_cv = GridSearchCV(rf_clf, param_grid = params, cv = 3, n_jobs = -1)
# grid_cv.fit(train_x, train_y)

# print('최적 하이퍼 파라미터: ', grid_cv.best_params_)
# print('최고 예측 정확도: {:.4f}'.format(grid_cv.best_score_))


In [20]:
pre_cal_rf = RandomForestClassifier(n_estimators = 30, 
                                               max_depth = 15, 
                                               min_samples_leaf = 8,
                                               min_samples_split = 20,
                                               n_jobs = -1,
                                               class_weight='balanced',
                                               random_state = RANDOM_STATE).fit(train_x, train_y)

In [21]:
rfc = CalibratedClassifierCV(pre_cal_rf, method='isotonic', cv=5)
rfc.fit(train_x, train_y)

CalibratedClassifierCV(cv=5,
                       estimator=RandomForestClassifier(class_weight='balanced',
                                                        max_depth=15,
                                                        min_samples_leaf=8,
                                                        min_samples_split=20,
                                                        n_estimators=30,
                                                        n_jobs=-1,
                                                        random_state=110),
                       method='isotonic')

### 모델 학습 (SVM)


In [22]:
pre_cal_svc = SVC(probability=True, 
                  random_state=RANDOM_STATE, 
                  class_weight='balanced')

In [23]:
svc = CalibratedClassifierCV(pre_cal_svc, method='isotonic', cv=7)
svc.fit(train_x, train_y)

CalibratedClassifierCV(cv=7,
                       estimator=SVC(class_weight='balanced', probability=True,
                                     random_state=110),
                       method='isotonic')

### 모델학습 (CatBoostClassifier)

In [24]:
# pre_cal_cat = CatBoostClassifier(
#     iterations=1200,
#     learning_rate=0.1, 
#     depth=6, 
#     loss_function='Logloss',
#     verbose=True,
#     eval_metric='Accuracy',
#     random_seed=RANDOM_STATE
# )

In [25]:
# cbc = CalibratedClassifierCV(pre_cal_cat, method='isotonic', cv=5)
# cbc.fit(train_x, train_y)

0:	learn: 0.5843465	total: 57.1ms	remaining: 1m 8s
1:	learn: 0.5847264	total: 67.5ms	remaining: 40.5s
2:	learn: 0.6041033	total: 76.3ms	remaining: 30.5s
3:	learn: 0.6010638	total: 84.8ms	remaining: 25.4s
4:	learn: 0.6003040	total: 92ms	remaining: 22s
5:	learn: 0.6128419	total: 98.5ms	remaining: 19.6s
6:	learn: 0.6189210	total: 105ms	remaining: 17.8s
7:	learn: 0.6155015	total: 111ms	remaining: 16.5s
8:	learn: 0.6200608	total: 115ms	remaining: 15.2s
9:	learn: 0.6212006	total: 120ms	remaining: 14.3s
10:	learn: 0.6193009	total: 124ms	remaining: 13.4s
11:	learn: 0.6200608	total: 128ms	remaining: 12.7s
12:	learn: 0.6215805	total: 132ms	remaining: 12s
13:	learn: 0.6215805	total: 136ms	remaining: 11.5s
14:	learn: 0.6200608	total: 140ms	remaining: 11s
15:	learn: 0.6189210	total: 144ms	remaining: 10.6s
16:	learn: 0.6185410	total: 146ms	remaining: 10.2s
17:	learn: 0.6181611	total: 150ms	remaining: 9.85s
18:	learn: 0.6200608	total: 154ms	remaining: 9.57s
19:	learn: 0.6212006	total: 158ms	remaining

CalibratedClassifierCV(cv=5,
                       estimator=<catboost.core.CatBoostClassifier object at 0x7fbcf64cc7f0>,
                       method='isotonic')

### Voting Ensemble

In [26]:
model = VotingClassifier(
    estimators = [('Random Forest Classifier', rfc), ('XGBoosting Classifier', xgbc), ('svm', svc)], voting='soft', weights=[5,2,1])

model.fit(train_x, train_y)

VotingClassifier(estimators=[('Random Forest Classifier',
                              CalibratedClassifierCV(cv=5,
                                                     estimator=RandomForestClassifier(class_weight='balanced',
                                                                                      max_depth=15,
                                                                                      min_samples_leaf=8,
                                                                                      min_samples_split=20,
                                                                                      n_estimators=30,
                                                                                      n_jobs=-1,
                                                                                      random_state=110),
                                                     method='isotonic')),
                             ('XGBoosting Classifier',
                              CalibratedClassifierCV(cv=5,
                                                     estimator=XGBClassifier(base_score=None,
                                                                             b...
                                                                             max_depth=3,
                                                                             max_leaves=None,
                                                                             min_child_weight=4,
                                                                             missing=nan,
                                                                             monotone_constraints=None,
                                                                             multi_strategy=None,
                                                                             n_estimators=300,
                                                                             n_jobs=-1,
                                                                             num_parallel_tree=None,
                                                                             random_state=110, ...),
                                                     method='isotonic')),
                             ('svm',
                              CalibratedClassifierCV(cv=7,
                                                     estimator=SVC(class_weight='balanced',
                                                                   probability=True,
                                                                   random_state=110),
                                                     method='isotonic'))],
                 voting='soft', weights=[5, 2, 1])

In [27]:
voting_res = model.predict(test_x)
encoded_voting_res = le.fit_transform(voting_res)
print(f1_score(encoded_y_test, encoded_voting_res))

0.604267033723331


## 4. 제출하기


### 테스트 데이터 예측


테스트 데이터 불러오기


In [28]:
test_data = pd.read_csv(os.path.join(ROOT_DIR, "test.csv"))

In [29]:
df_test_x = test_data[features]

for col in df_test_x.columns:
    try:
        df_test_x.loc[:, col] = df_test_x[col].astype(int)
    except:
        continue

In [30]:
test_pred = model.predict(df_test_x)
test_pred

array(['AbNormal', 'Normal', 'AbNormal', ..., 'Normal', 'AbNormal',
       'Normal'], dtype=object)

### 제출 파일 작성


In [31]:
# 제출 데이터 읽어오기 (df_test는 전처리된 데이터가 저장됨)
df_sub = pd.read_csv("submission.csv")
df_sub["target"] = test_pred

# 제출 파일 저장
df_sub.to_csv("submission.csv", index=False)

**우측 상단의 제출 버튼을 클릭해 결과를 확인하세요**
